# 예제 03. BatchNorm 적용
빅데이터프로그래밍 · 9주차

## 목표
- BatchNorm이 각 층의 입력 분포를 정리하는 것을 확인한다
- 학습이 빨라지는 것을 곡선으로 본다
- 넣는 위치를 안다 (Conv 다음, ReLU 앞)

각 층의 입력 분포를 안정적으로 만들어 학습을 돕습니다.


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import pandas as pd

torch.manual_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)


## 1. 분포를 정리한다는 것
평균 0, 표준편차 1 근처로 맞춥니다. 3주차 표준화와 같은 계산을 층마다 하는 것입니다.


In [ ]:
x = torch.randn(64, 32) * 5 + 10        # 평균 10, 표준편차 5쯤
bn = nn.BatchNorm1d(32)
bn.train()
out = bn(x)

print(f"입력  평균 {x.mean():7.4f}  표준편차 {x.std():7.4f}")
print(f"출력  평균 {out.mean():7.4f}  표준편차 {out.std():7.4f}")


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.2))
ax[0].hist(x.flatten(), bins=40); ax[0].set_title("BatchNorm 전")
ax[1].hist(out.detach().flatten(), bins=40); ax[1].set_title("BatchNorm 후")
plt.tight_layout(); plt.show()


## 2. 층이 깊어질 때 분포가 흐트러지는 문제


In [ ]:
def track(use_bn):
    layers, stats = [], []
    h = torch.randn(128, 64)
    for i in range(6):
        lin = nn.Linear(64, 64)
        h = lin(h)
        if use_bn:
            b = nn.BatchNorm1d(64); b.train(); h = b(h)
        h = torch.relu(h)
        stats.append((h.mean().item(), h.std().item()))
    return stats


torch.manual_seed(0); no_bn = track(False)
torch.manual_seed(0); with_bn = track(True)

df = pd.DataFrame({
    "층": range(1, 7),
    "BN 없음 · 표준편차": [round(s[1], 4) for s in no_bn],
    "BN 있음 · 표준편차": [round(s[1], 4) for s in with_bn],
})
print(df.to_string(index=False))


BN이 없으면 층을 지날수록 값의 크기가 계속 변합니다. BN은 매 층에서 다시 정리합니다.


## 3. 넣는 위치 — Conv 다음, ReLU 앞


In [ ]:
block = nn.Sequential(
    nn.Conv2d(1, 32, 3, padding=1),
    nn.BatchNorm2d(32),          # Conv의 출력 채널 수와 같아야 합니다
    nn.ReLU(),
    nn.MaxPool2d(2),
)
print(block)

x = torch.randn(8, 1, 28, 28)
print("\n출력:", tuple(block(x).shape))


In [ ]:
# 채널 수를 틀리면 오류
try:
    nn.Sequential(nn.Conv2d(1, 32, 3), nn.BatchNorm2d(16))(x)
except RuntimeError as err:
    print("RuntimeError:", err)


## 4. 데이터와 공통 함수


In [ ]:
transform = transforms.ToTensor()
full_train = datasets.FashionMNIST("./data", train=True,  download=True, transform=transform)
test_set   = datasets.FashionMNIST("./data", train=False, download=True, transform=transform)

train_loader = DataLoader(Subset(full_train, range(2000)), batch_size=64, shuffle=True)
test_loader  = DataLoader(test_set, batch_size=256, shuffle=False)

loss_fn = nn.CrossEntropyLoss()

def measure(model, loader):
    model.eval()
    loss_sum = correct = total = 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            loss_sum += loss_fn(out, y).item() * y.numel()
            correct += (out.argmax(dim=1) == y).sum().item()
            total += y.numel()
    return loss_sum / total, correct / total


def train(model, epochs=30, lr=1e-3):
    model = model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    hist = []
    for epoch in range(epochs):
        model.train()
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            loss = loss_fn(model(x), y)
            opt.zero_grad(); loss.backward(); opt.step()
        hist.append((*measure(model, train_loader), *measure(model, test_loader)))
    return model, hist


## 5. BatchNorm 있고 없고 비교


In [ ]:
class PlainCNN(nn.Module):
    def __init__(self, n_classes=10):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Flatten(),
            nn.Linear(64*7*7, 256), nn.ReLU(),
            nn.Linear(256, n_classes),
        )
    def forward(self, x):
        return self.net(x)


class BNCNN(nn.Module):
    def __init__(self, n_classes=10):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Flatten(),
            nn.Linear(64*7*7, 256), nn.BatchNorm1d(256), nn.ReLU(),
            nn.Linear(256, n_classes),
        )
    def forward(self, x):
        return self.net(x)


torch.manual_seed(42); plain, plain_hist = train(PlainCNN())
torch.manual_seed(42); bnm, bn_hist = train(BNCNN())

print("기본  최종 검증 정확도:", round(plain_hist[-1][3], 4))
print("BN    최종 검증 정확도:", round(bn_hist[-1][3], 4))


In [ ]:
xs = range(1, len(plain_hist) + 1)
fig, ax = plt.subplots(1, 2, figsize=(13, 4.2))
ax[0].plot(xs, [h[0] for h in plain_hist], label="기본")
ax[0].plot(xs, [h[0] for h in bn_hist],  label="BatchNorm")
ax[0].set_title("학습 손실 — BN이 더 빨리 내려갑니다"); ax[0].set_xlabel("epoch"); ax[0].legend(); ax[0].grid(alpha=.3)
ax[1].plot(xs, [h[3] for h in plain_hist], label="기본")
ax[1].plot(xs, [h[3] for h in bn_hist],  label="BatchNorm")
ax[1].set_title("검증 정확도"); ax[1].set_xlabel("epoch"); ax[1].legend(); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()


## 6. 학습률을 크게 줘도 견딥니다
BN의 실질적인 장점입니다. 학습률 0.01은 보통 불안정한 값입니다.


In [ ]:
rows = []
for lr in [1e-3, 1e-2]:
    for name, cls in [("기본", PlainCNN), ("BatchNorm", BNCNN)]:
        torch.manual_seed(42)
        _, h = train(cls(), epochs=15, lr=lr)
        rows.append({"모델": name, "학습률": lr, "검증 정확도": round(h[-1][3], 4)})
pd.DataFrame(rows)


## 직접 해보기
1. BatchNorm을 ReLU **뒤에** 넣으면 결과가 달라지나요?
2. batch_size를 8로 줄이면 BN의 효과가 어떻게 되나요? (BN은 batch가 작으면 불안정합니다)


In [ ]:
# 여기에 작성하세요
